# CP5 - Operaciones Avanzadas I

### Como siempre, antes de empezar...

Si has cerrado el proyecto o has abierto `Nueva Ventana` en VS Code, pulsa en el primer icono del menú de la izquierda `Explorador` y después en `Abrir Carpeta`. 

Busca la ruta en la que guardas el repositorio de GitHub en tu PC, normalmente *`Documentos => GitHub => fundamentos_big_data` * y pulsa en `Abrir`. De esta forma, ya tendrás la carpeta de tu repositorio cargada en VS Code.

> ¡Importante! Antes de abrir esa carpeta, debes ir a GitHub Desktop y realizar un `pull` para asegurarte de que tus archivos están actualizados. También puedes hacer el pull desde VS Code.  

### Requisitos para arrancar el entonro
- Iniciar Docker (abrir docker desktop).
- Iniciar GitHub (abrir github desktop) y estar logueado.
- En el terminal, ejecutar el siguiente comando en la raíz del proyecto `fundamentos_big_data %` 
```bash
docker-compose up --build
```
> ¡NO TE OLVIDES! Docker debe estar abierto y en ejecución en tu pc para poder utilizarlo. 


### Seleccionar Kernel
- Seleccionar kernel
- Servidor de Juypter existente
- Escriba la dirección URL del servidor de Jupyter en ejecución
- Seleccionar http://127.0.0.1:8888/?token=xxxxxxxx y pulsar Enter
- 127.0.0.1 y pulsar Enter
- Seleccionar Python 3 (ipykernel)

___________________

## Objetivos de la sesión

- Funciones de agregación con agrupación.
- Integrar datos mediante el uso de joins.
- Conocer y aplicar funciones de fecha y hora.

In [ ]:
### Importar las librerías necesarias ###
### INICIA LA SPARK-SESSION ###
### Añade el spark context ###
### Añade el path para acceder a los datos y nómbralo como DATA_PATH###
### Lee la carpeta vuelos con pyspark y nómbralo como df_v ###

> **Importante**: a lo largo del notebook hay código que está mal o no funciona. Tendrás que modificarlo para que funcione. Por ejemplo, si cuando importes las funciones de pyspark como F, deberás añadir `F.` a las funciones que lo necesiten, como `col` o `desc`.

## 2 - Dataframes en PySpark

### 2.10 Agregación con Agrupación

El primer paso es agrupar usando la transformación `groupBy(<col1>,<col2>,...)`, especificando por qué columnas agrupamos las filas. El segundo paso es añadir las funciones de agregación deseadas.

#### **Función `groupBy()`**

Agrupa las filas en función de las columnas seleccionadas. Está supeditada a que se le aplique una función de agregación como `count()`,`min()`,`max()`, etc.

In [ ]:
(df_v.groupBy('ORIGIN_AIRPORT')
    .count()
    .orderBy(desc('count'))
).show(10)

+--------------+------+
|ORIGIN_AIRPORT| count|
+--------------+------+
|           ATL|346836|
|           ORD|285884|
|           DFW|239551|
|           DEN|196055|
|           LAX|194673|
|           SFO|148008|
|           PHX|146815|
|           IAH|146622|
|           LAS|133181|
|           MSP|112117|
+--------------+------+
only showing top 10 rows



In [ ]:
(df_v.groupBy('ORIGIN_AIRPORT', 'DESTINATION_AIRPORT')
    .count()
    .orderBy(desc('count'))
).show(10)

+--------------+-------------------+-----+
|ORIGIN_AIRPORT|DESTINATION_AIRPORT|count|
+--------------+-------------------+-----+
|           SFO|                LAX|13744|
|           LAX|                SFO|13457|
|           JFK|                LAX|12016|
|           LAX|                JFK|12015|
|           LAS|                LAX| 9715|
|           LGA|                ORD| 9639|
|           LAX|                LAS| 9594|
|           ORD|                LGA| 9575|
|           SFO|                JFK| 8440|
|           JFK|                SFO| 8437|
+--------------+-------------------+-----+
only showing top 10 rows



#### **Función `agg()`**

Nos permite realizar **múltiples agregaciones** por grupo en lugar de una.

In [ ]:
df_v.groupBy('ORIGIN_AIRPORT').agg(
    count('AIR_TIME').alias('tiempo_aire'),
    min('AIR_TIME').alias('minimo'),
    max('AIR_TIME').alias('maximo')
).orderBy(desc('tiempo_aire')).show(10)

+--------------+-----------+------+------+
|ORIGIN_AIRPORT|tiempo_aire|minimo|maximo|
+--------------+-----------+------+------+
|           ATL|     343506|    15|   614|
|           ORD|     276554|    13|   571|
|           DFW|     232647|    11|   534|
|           DEN|     193402|    12|   493|
|           LAX|     192003|    14|   409|
|           PHX|     145552|    19|   444|
|           SFO|     145491|     8|   389|
|           IAH|     144019|    15|   524|
|           LAS|     131937|    25|   429|
|           MSP|     111055|    14|   537|
+--------------+-----------+------+------+
only showing top 10 rows



In [ ]:
df_v.groupBy('MONTH').agg(
    count('ARRIVAL_DELAY').alias('conteo_retrasos'),
    avg('DISTANCE').alias('promedio_dist')
).orderBy(desc('conteo_retrasos')).show(10)

+-----+---------------+-----------------+
|MONTH|conteo_retrasos|    promedio_dist|
+-----+---------------+-----------------+
|    7|         514384|841.4772794487611|
|    8|         503956|834.8244276603413|
|    6|         492847|835.6302716626612|
|    3|         492138|816.0553268611494|
|    5|         489641|823.3230588760807|
|   10|         482878|816.4436127652134|
|    4|         479251|817.0060476016745|
|   12|         469717|837.8018926194103|
|   11|         462367|820.2482434846529|
|    9|         462153|815.8487523282274|
+-----+---------------+-----------------+
only showing top 10 rows



#### **Función `pivot()`**

Nos permite realizar una agregación **pivotando las filas por las columnas**. Como parámetros se le puede añadir la columna sobre la que se pivota y entre corchetes `[]` podemos poner alguna de las categorías de esa variable.

In [ ]:
df_e = spark.read.parquet('./data/estudiantes.parquet')

In [ ]:
df_e.show()

+------+----+----+----------+
|nombre|sexo|peso|graduacion|
+------+----+----+----------+
|  Jose|   M|  80|      2000|
| Hilda|   F|  50|      2000|
|  Juan|   M|  75|      2000|
| Pedro|   M|  76|      2001|
|Katia+|   F|  65|      2001|
+------+----+----+----------+



In [ ]:
df_e.groupBy('graduacion').pivot('sexo').agg(
    avg('peso')
).show(5)

+----------+----+----+
|graduacion|   F|   M|
+----------+----+----+
|      2001|65.0|76.0|
|      2000|50.0|77.5|
+----------+----+----+



In [ ]:
df_e.groupBy('graduacion').pivot('sexo', ['M']).agg(
    avg('peso')
).show(5)

+----------+----+
|graduacion|   M|
+----------+----+
|      2001|76.0|
|      2000|77.5|
+----------+----+



### 2.11 Joins

Se trata de una técnica de combinación de tablas usando una columna común entre ellas para obtener como resultado un dataset que es combinación de ellos. Existen distintos tipos de Joins:

- **Inner Join** (por defecto en Spark): elementos comunes entre A y B.
- **Left Outer Join**: elementos de A y los compartidos con B.
- **Right Outer Join**: elementos de B y los compartidos con A.
- **Outer Join**: elementos de A, de B y de la combinación de ambos.
- **Left anti Join**: elementos de A que no están en B (diferencia).
- **Left Semi Join**: elementos de A y compartidos con B, pero solo muestra los de A.
- **Cross Join**: producto escalar de A y B.

La estructura fundamental de todos los métodos `join` es:

```python
df1.join(df2, <coldf1> == <coldf2>, '<join_type>')
```

Alternativamente, tenemos el `crossJoin`:

```python
df1.crossJoin(df2)
```

In [ ]:
emple = spark.read.parquet(DATA_PATH + 'empleados.parquet')
depart = spark.read.parquet(DATA_PATH + 'departamentos.parquet')

In [ ]:
emple.show()
depart.show()

+------+--------+
|nombre|num_dpto|
+------+--------+
|  Luis|      33|
| Katia|      33|
|  Raul|      34|
| Pedro|       0|
| Laura|      34|
|Sandro|      31|
+------+--------+

+---+-----------+
| id|nombre_dpto|
+---+-----------+
| 31|     letras|
| 33|    derecho|
| 34| matemática|
| 35|informática|
+---+-----------+



#### Inner Join

In [ ]:
df8 = emple.join(depart, col('num_dpto') == col('id')) # por defecto es inner
df8.show()

+------+--------+---+-----------+
|nombre|num_dpto| id|nombre_dpto|
+------+--------+---+-----------+
|  Luis|      33| 33|    derecho|
| Katia|      33| 33|    derecho|
|  Raul|      34| 34| matemática|
| Laura|      34| 34| matemática|
|Sandro|      31| 31|     letras|
+------+--------+---+-----------+



In [ ]:
# Forma alternativa usando where()
emple.join(depart).where(col('num_dpto') == col('id')).show()

+------+--------+---+-----------+
|nombre|num_dpto| id|nombre_dpto|
+------+--------+---+-----------+
|  Luis|      33| 33|    derecho|
| Katia|      33| 33|    derecho|
|  Raul|      34| 34| matemática|
| Laura|      34| 34| matemática|
|Sandro|      31| 31|     letras|
+------+--------+---+-----------+



#### Left Outer Join

Se puede usar: `leftouter`, `left` o `left_outer`

In [ ]:
emple.join(depart, col('num_dpto') == col('id'), 'leftouter').show()

+------+--------+----+-----------+
|nombre|num_dpto|  id|nombre_dpto|
+------+--------+----+-----------+
|  Luis|      33|  33|    derecho|
| Katia|      33|  33|    derecho|
|  Raul|      34|  34| matemática|
| Pedro|       0|null|       null|
| Laura|      34|  34| matemática|
|Sandro|      31|  31|     letras|
+------+--------+----+-----------+



#### Right Outer Join

Se puede usar: `rightouter`, `right` o `right_outer`

In [ ]:
emple.join(depart, col('num_dpto') == col('id'), 'rightouter').show()

+------+--------+---+-----------+
|nombre|num_dpto| id|nombre_dpto|
+------+--------+---+-----------+
|Sandro|      31| 31|     letras|
| Katia|      33| 33|    derecho|
|  Luis|      33| 33|    derecho|
| Laura|      34| 34| matemática|
|  Raul|      34| 34| matemática|
|  null|    null| 35|informática|
+------+--------+---+-----------+



#### Outer Join

In [ ]:
emple.join(depart, col('num_dpto') == col('id'), 'outer').show()

+------+--------+----+-----------+
|nombre|num_dpto|  id|nombre_dpto|
+------+--------+----+-----------+
| Pedro|       0|null|       null|
|Sandro|      31|  31|     letras|
|  Luis|      33|  33|    derecho|
| Katia|      33|  33|    derecho|
|  Raul|      34|  34| matemática|
| Laura|      34|  34| matemática|
|  null|    null|  35|informática|
+------+--------+----+-----------+



#### Left Anti Join

Se puede usar: `left_anti` o `anti`

In [ ]:
emple.join(depart, col('num_dpto') == col('id'), 'anti').show()

+------+--------+
|nombre|num_dpto|
+------+--------+
| Pedro|       0|
+------+--------+



Cambiando el orden de los dataframes, obtendremos los elementos que están en `depart` y no están en `emple`.

In [ ]:
depart.join(emple, col('num_dpto') == col('id'), 'anti').show()

+---+-----------+
| id|nombre_dpto|
+---+-----------+
| 35|informática|
+---+-----------+



#### Left Semi Join

Se puede usar: `left_semi` o `semi`

In [ ]:
emple.join(depart, col('num_dpto') == col('id'), 'semi').show()

+------+--------+
|nombre|num_dpto|
+------+--------+
|  Luis|      33|
| Katia|      33|
|  Raul|      34|
| Laura|      34|
|Sandro|      31|
+------+--------+



#### Cross Join

In [ ]:
emple.crossJoin(depart).show()

+------+--------+---+-----------+
|nombre|num_dpto| id|nombre_dpto|
+------+--------+---+-----------+
|  Luis|      33| 31|     letras|
|  Luis|      33| 33|    derecho|
|  Luis|      33| 34| matemática|
|  Luis|      33| 35|informática|
| Katia|      33| 31|     letras|
| Katia|      33| 33|    derecho|
| Katia|      33| 34| matemática|
| Katia|      33| 35|informática|
|  Raul|      34| 31|     letras|
|  Raul|      34| 33|    derecho|
|  Raul|      34| 34| matemática|
|  Raul|      34| 35|informática|
| Pedro|       0| 31|     letras|
| Pedro|       0| 33|    derecho|
| Pedro|       0| 34| matemática|
| Pedro|       0| 35|informática|
| Laura|      34| 31|     letras|
| Laura|      34| 33|    derecho|
| Laura|      34| 34| matemática|
| Laura|      34| 35|informática|
+------+--------+---+-----------+
only showing top 20 rows



> **Nota importante**: si encontramos dos columnas con el mismo nombre a la hora de hacer un `join`, es fundamental renombrar alguna de las columnas o hacer uso de la estructura `<df>[<'columna'>]`

### 2.12 Funciones de Fecha y Hora

In [ ]:
df_f = spark.read.parquet(DATA_PATH + 'convertir')

In [ ]:
df_f.printSchema()

root
 |-- date: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- date_str: string (nullable = true)
 |-- ts_str: string (nullable = true)



In [ ]:
df_f.show(truncate=False)

+----------+-----------------------+----------+----------------+
|date      |timestamp              |date_str  |ts_str          |
+----------+-----------------------+----------+----------------+
|2021-01-01|2021-01-01 20:10:50.723|01-01-2021|18-08-2021 46:58|
+----------+-----------------------+----------+----------------+



#### **Funcion `to_date()` y `to_timestamp`**

Sirven para cambiar a tipo fecha y a tipo timestamp los string con valores de fecha, especificando la forma en la que se recibe ese patrón del string (ej: dd-MM-yyyy).

In [ ]:
df_fecha = df_f.select(
    to_date(col('date')).alias('fecha'),
    to_timestamp(col('timestamp')).alias('reg_tiempo'),
    to_date(col('date_str'), 'dd-MM-yyyy').alias('fecha_2'),
    to_timestamp(col('ts_str'), 'dd-MM-yyyy mm:ss').alias('reg_tiempo_2')
)
df_fecha.show(truncate=False)

+----------+-----------------------+----------+-------------------+
|fecha     |reg_tiempo             |fecha_2   |reg_tiempo_2       |
+----------+-----------------------+----------+-------------------+
|2021-01-01|2021-01-01 20:10:50.723|2021-01-01|2021-08-18 00:46:58|
+----------+-----------------------+----------+-------------------+



In [ ]:
df_fecha.printSchema()

root
 |-- fecha: date (nullable = true)
 |-- reg_tiempo: timestamp (nullable = true)
 |-- fecha_2: date (nullable = true)
 |-- reg_tiempo_2: timestamp (nullable = true)



#### **Función `date_format()`**

Nos sirve para cambiar de un formato de fecha a otro, por ejemplo: la columna `fecha` está en formato `yyyy-MM-dd`. Con `date_format()` podemos cambiarlo a `dd-MM-yyyy`.

In [ ]:
df_fecha.select(
    date_format(col('fecha'), 'dd-MM-yyyy').alias('fecha dd-MM-yyyy')
).show()

+----------------+
|fecha dd-MM-yyyy|
+----------------+
|      01-01-2021|
+----------------+



In [ ]:
df_calc = spark.read.parquet(DATA_PATH + 'calculo.parquet')

In [ ]:
df_calc.show()

+------+-------------+------------+-------------------+
|nombre|fecha_ingreso|fecha_salida|       baja_sistema|
+------+-------------+------------+-------------------+
|  Jose|   2021-01-01|  2021-11-14|2021-10-14 15:35:59|
|Mayara|   2021-02-06|  2021-11-25|2021-11-25 10:35:55|
+------+-------------+------------+-------------------+



In [ ]:
df_calc.printSchema()

root
 |-- nombre: string (nullable = true)
 |-- fecha_ingreso: string (nullable = true)
 |-- fecha_salida: string (nullable = true)
 |-- baja_sistema: string (nullable = true)



#### **Funciones `datediff()`, `months_between()`y `last_day()`**

Nos permite saber los días que pasan entre dos fechas, los meses que pasan entre dos fechas y el último día de alguna fecha, respectivamente (funcionan con strings).


In [ ]:
df_calc.select(
    datediff(col('fecha_salida'), col('fecha_ingreso')).alias('dias'),
    months_between(col('fecha_salida'), col('fecha_ingreso')).alias('meses'),
    last_day(col('fecha_salida')).alias('ultimo_dia')
).show()

+----+-----------+----------+
|dias|      meses|ultimo_dia|
+----+-----------+----------+
| 317|10.41935484|2021-11-30|
| 292| 9.61290323|2021-11-30|
+----+-----------+----------+



#### **Funciones `date_add()` y `date_sub()`**

Nos permiten añadir y eliminar días a una fecha determinada (funcionan con strings).

In [ ]:
df_calc.select(
    col('nombre'), col('fecha_ingreso'),
    date_add(col('fecha_ingreso'), 14).alias('mas_14_dias'),
    date_sub(col('fecha_ingreso'), 1).alias('memos_1_dia')
).show()

+------+-------------+-----------+-----------+
|nombre|fecha_ingreso|mas_14_dias|memos_1_dia|
+------+-------------+-----------+-----------+
|  Jose|   2021-01-01| 2021-01-15| 2020-12-31|
|Mayara|   2021-02-06| 2021-02-20| 2021-02-05|
+------+-------------+-----------+-----------+



#### **Funciones `year()`, `month()`, `dayofmonth()`, `dayofyear()`, `hour()`, `minute()` y `second()`**

Nos permite obtener esos valores en concreto, extrayéndolos de una fecha (funcionan con strings).

In [ ]:
df_calc.select(
    col('baja_sistema'),
    year('baja_sistema'),
    month('baja_sistema'),
    dayofmonth('baja_sistema'),
    dayofyear('baja_sistema'),
    hour('baja_sistema'),
    minute('baja_sistema'),
    second('baja_sistema')
).show()

+-------------------+------------------+-------------------+------------------------+-----------------------+------------------+--------------------+--------------------+
|       baja_sistema|year(baja_sistema)|month(baja_sistema)|dayofmonth(baja_sistema)|dayofyear(baja_sistema)|hour(baja_sistema)|minute(baja_sistema)|second(baja_sistema)|
+-------------------+------------------+-------------------+------------------------+-----------------------+------------------+--------------------+--------------------+
|2021-10-14 15:35:59|              2021|                 10|                      14|                    287|                15|                  35|                  59|
|2021-11-25 10:35:55|              2021|                 11|                      25|                    329|                10|                  35|                  55|
+-------------------+------------------+-------------------+------------------------+-----------------------+------------------+-----------------

### 2.13 Funciones para Strings

In [ ]:
df_str = spark.createDataFrame([(' Spark ',)], ['nombre'])
df_str.show()

+-------+
| nombre|
+-------+
| Spark |
+-------+



> **Nota**: el valor 'Spark' de la columna "nombre", tiene un espacio en blanco a la izquierda y otro a la derecha.

#### **Funciones `ltrim()`, `rtrim()` y `trim()`**

Son el equivalente en python a `lstrip()`, `rstrip()` y `strip()`.

In [ ]:
df_str.select(
    ltrim('nombre').alias('ltrim'),
    rtrim('nombre').alias('rtrim'),
    trim('nombre').alias('trim')
).show()

+------+------+-----+
| ltrim| rtrim| trim|
+------+------+-----+
|Spark | Spark|Spark|
+------+------+-----+



#### **Funciones `lpad()` y `rpad()`**

Sirve para rellenar a la izquierda o a la derecha con caracteres alfanuméricos. Se le pasa como parámetros:

- Columna que queremos seleccionar.
- Número total de caracteres que tendrá tras la transformación.
- Caracter de relleno.

In [ ]:
df_str.select(
    trim('nombre').alias('trim')
).select(
    lpad(col('trim'), 8, '-').alias('lpad'),
    rpad(col('trim'), 8, '=').alias('rpad')
).show()

+--------+--------+
|    lpad|    rpad|
+--------+--------+
|---Spark|Spark===|
+--------+--------+



#### **Funciones `concat_ws()`, `lower()`, `upper()`, `initcap()` y `reverse()`**

La función `concat_ws` tiene dos parámetros:

- Caracter de separación.
- Columna a la que aplicarlo.

In [ ]:
df_str2 = spark.createDataFrame([('Spark','es','maravilloso')],
                                ['sujeto', 'verbo', 'adjetivo'])
df_str2.show()

+------+-----+-----------+
|sujeto|verbo|   adjetivo|
+------+-----+-----------+
| Spark|   es|maravilloso|
+------+-----+-----------+



In [ ]:
df_str2.select(
    concat_ws(' ', col('sujeto'),col('verbo'),col('adjetivo')).alias('frase'),
).select(
    col('frase'),
    lower(col('frase')).alias('minuscula'),
    upper(col('frase')).alias('mayuscula'),
    initcap(col('frase')).alias('capital'),
    reverse(col('frase')).alias('reverse')
).show()

+--------------------+--------------------+--------------------+--------------------+--------------------+
|               frase|           minuscula|           mayuscula|             capital|             reverse|
+--------------------+--------------------+--------------------+--------------------+--------------------+
|Spark es maravilloso|spark es maravilloso|SPARK ES MARAVILLOSO|Spark Es Maravilloso|osollivaram se krapS|
+--------------------+--------------------+--------------------+--------------------+--------------------+



#### **Función `regexp_replace()`**

La función tiene 3 parámetros principales:

- Columna a la que aplicaremos cambios.
- Término(s) que queremos sustituir.
- Término que se añadirá.

In [ ]:
df_str3 = spark.createDataFrame([('voy a casa a por mis llaves',)], ['frase'])

In [ ]:
df_str3.select(
    regexp_replace(col('frase'), 'voy|por', 'ir').alias('nueva_frase')
).show(truncate=False)

+-------------------------+
|nueva_frase              |
+-------------------------+
|ir a casa a ir mis llaves|
+-------------------------+



### 2.14 Funciones para trabajar con Colecciones

In [ ]:
df_c = spark.read.parquet(DATA_PATH + 'colecciones')

In [ ]:
df_c.show(truncate=False)

+-----+--------------------------------------------+
|dia  |tareas                                      |
+-----+--------------------------------------------+
|lunes|[hacer la tarea, buscar agua, lavar el auto]|
+-----+--------------------------------------------+



In [ ]:
df_c.printSchema()

root
 |-- dia: string (nullable = true)
 |-- tareas: array (nullable = true)
 |    |-- element: string (containsNull = true)



En este caso, tenemos un Schema que nos muestra que una de las variables es de tipo `array` y que incluye una colección de elementos de tipo `str`.

#### **Función `size()`, `sort_array()` y `array_contains()`**

In [ ]:
df_c.select(
    size(col('tareas')).alias('size'),
    sort_array(col('tareas')).alias('array_ordenado'),
    array_contains(col('tareas'), 'buscar agua').alias('contiene...')
).show(truncate=False)

+----+--------------------------------------------+-----------+
|size|array_ordenado                              |contiene...|
+----+--------------------------------------------+-----------+
|3   |[buscar agua, hacer la tarea, lavar el auto]|true       |
+----+--------------------------------------------+-----------+



#### **Función `explode()`**

Nos permite "deconstruir" un array en filas de una columna seleccionada.

In [ ]:
df_c.select(
    col('dia'),
    explode(col('tareas')).alias('tareas')
).show()

+-----+--------------+
|  dia|        tareas|
+-----+--------------+
|lunes|hacer la tarea|
|lunes|   buscar agua|
|lunes| lavar el auto|
+-----+--------------+



### 2.15 Otras funciones

#### **Función `when()`**

La función `when()` sirve para establecer condiciones. Se le introducen estos parámetros:

- Columna con condición lógica.
- Salida de esa condición.

Se puede añadir un valor por defecto usando la función `otherwise()`.

In [ ]:
df_p = spark.createDataFrame([('Jose',1),('Julia',2),('Katia',1),
                              (None, 3),('Raul',3)], ['nombre','pago'])
df_p.show()

+------+----+
|nombre|pago|
+------+----+
|  Jose|   1|
| Julia|   2|
| Katia|   1|
|  null|   3|
|  Raul|   3|
+------+----+



In [ ]:
df_p.select(
    col('nombre'),
    (when(col('pago') == 1, 'pagado')
    .when(col('pago') == 2, 'no pagado')
    .otherwise('sin iniciar')).alias('status')
).show()

+------+-----------+
|nombre|     status|
+------+-----------+
|  Jose|     pagado|
| Julia|  no pagado|
| Katia|     pagado|
|  null|sin iniciar|
|  Raul|sin iniciar|
+------+-----------+



#### **Funciones `coalesce()` y `lit()`**

La función `coalesce()` va a recibir `n` columnas y va a devolver el **primer valor que no sea nulo**. Por su parte, `lit()` nos permite añadir un literal. En el ejemplo, como tenemos un valor a `null`, lo vamos a detectar con `coalesce()` y luego renombrarlo usando `lit()`.

In [ ]:
df_p.select(
    coalesce(col('nombre'), lit('sin nombre')).alias('nombre')
).show()

+----------+
|    nombre|
+----------+
|      Jose|
|     Julia|
|     Katia|
|sin nombre|
|      Raul|
+----------+

